# Text and Categorical Cleaning (Advanced)

Welcome to the messy reality of human-generated data. While numbers are exact, text is chaotic. People make typos, use different capitalizations, add extra spaces, and write in unstructured sentences. If you do not clean your text and categorical features perfectly, your machine learning model will treat "New York", "new york", and "New York " as three completely different cities!

In this lesson, we level up our Pandas skills to handle high-cardinality categories, extract hidden signals using Regular Expressions (Regex), and standardize human error.

Let's set up a Python sandbox with a deliberately chaotic dataset representing a customer feedback form. Notice how the cities have typos, the IDs are buried in text, and there are way too many unique job titles.

In [1]:
import pandas as pd
import numpy as np
import re

# Create a highly chaotic dataset
data = {
    'customer_name': [' Alice ', 'BOB', 'charlie', '  David', 'Eve'],
    'city': ['New York', 'new york', 'New Yrk', 'Los Angeles', 'L.A.'],
    'job_title': ['Software Engineer', 'Data Scientist', 'CEO', 'Janitor', 'Software Engineer'],
    'messy_notes': [
        'User id: 12345. Called to complain.',
        'Refund processed for ID 98765 on Tuesday.',
        'id:45678 upgrade requested',
        'No ID provided. Hung up.',
        'User-ID: 33333 loves the product!'
    ]
}

df = pd.DataFrame(data)

print("--- The Chaotic Raw Data ---")
display(df)

--- The Chaotic Raw Data ---


,customer_name,city,job_title,messy_notes
0,Alice,New York,Software Engineer,User id: 12345. Called to complain.
1,BOB,new york,Data Scientist,Refund processed for ID 98765 on Tuesday.
2,charlie,New Yrk,CEO,id:45678 upgrade requested
3,David,Los Angeles,Janitor,No ID provided. Hung up.
4,Eve,L.A.,Software Engineer,User-ID: 33333 loves the product!


# 1. Standardizing Strings (The Basics)
The very first step with any text data is to standardize the case and remove accidental whitespace. If a user accidentally hits the spacebar after typing their name, Pandas treats `"Alice"` and `"Alice "` as two completely different people.

We fix this using the `.str` accessor in Pandas, which unlocks specialized string manipulation tools.

In [2]:
# Create a copy
df_clean = df.copy()

# 1. Strip whitespace from the edges
df_clean['customer_name'] = df_clean['customer_name'].str.strip()

# 2. Standardize capitalization (Title Case)
df_clean['customer_name'] = df_clean['customer_name'].str.title()

# 3. Standardize city names to lowercase for easier matching later
df_clean['city'] = df_clean['city'].str.lower().str.strip()

print("--- Data after Basic Standardization ---")
display(df_clean[['customer_name', 'city']])

--- Data after Basic Standardization ---


,customer_name,city
0,Alice,new york
1,Bob,new york
2,Charlie,new yrk
3,David,los angeles
4,Eve,l.a.


# 2. Grouping Rare Categories (The "Other" Bucket)
If you have a column like `job_title` with 10,000 unique titles, applying One-Hot Encoding (from Lesson 05) will create 10,000 brand new columns! This is called **High Cardinality**, and it will completely crash your model's memory.

To fix this, we find the "Tail" (the rare categories that only appear a few times) and group them all into a single, generic `"Other"` bucket.

In [3]:
# Let's pretend we have a much larger dataset of jobs
large_jobs_data = pd.Series([
    'Engineer', 'Engineer', 'Engineer', 'Manager', 'Manager', 
    'CEO', 'Janitor', 'Barista', 'Pilot'
])

print("--- Original Job Counts ---")
print(large_jobs_data.value_counts())

# We want to keep any job that appears at least 2 times. 
# Everything else becomes "Other".

# 1. Find the frequencies
frequencies = large_jobs_data.value_counts()

# 2. Identify the categories to keep
categories_to_keep = frequencies[frequencies >= 2].index

# 3. Apply the filter using numpy's `where` statement
# "If the job is in our keep list, keep it. Otherwise, rename it to 'Other'"
cleaned_jobs = np.where(large_jobs_data.isin(categories_to_keep), large_jobs_data, 'Other')

print("\n--- Job Counts after 'Other' Bucket ---")
print(pd.Series(cleaned_jobs).value_counts())

--- Original Job Counts ---
Engineer    3
Manager     2
CEO         1
Janitor     1
Barista     1
Pilot       1
Name: count, dtype: int64

--- Job Counts after 'Other' Bucket ---
Other       4
Engineer    3
Manager     2
Name: count, dtype: int64


*(By doing this, we drastically reduced our unique categories from 5 down to 3, saving massive amounts of memory while keeping the most important predictive categories intact!)*

# 3. Regular Expressions (Regex) for Feature Extraction
Look at our `messy_notes` column. It is completely unstructured text. But hidden inside almost every row is a 5-digit Customer ID! 

We can extract this specific pattern using **Regular Expressions (Regex)**. Regex is a universal programming language used purely for pattern matching. 

* `\d` means "Any digit (0-9)"
* `{5}` means "Exactly five times in a row"
* Therefore, `\d{5}` means "Find me any 5-digit number."

In [4]:
# Extract the 5-digit ID using Pandas .str.extract() and a Regex pattern
# We use parentheses (\d{5}) to tell Pandas exactly what part to capture
df_clean['extracted_id'] = df_clean['messy_notes'].str.extract(r'(\d{5})')

print("--- Extracting Hidden Data with Regex ---")
display(df_clean[['messy_notes', 'extracted_id']])

--- Extracting Hidden Data with Regex ---


,messy_notes,extracted_id
0,User id: 12345. Called to complain.,12345
1,Refund processed for ID 98765 on Tuesday.,98765
2,id:45678 upgrade requested,45678
3,No ID provided. Hung up.,NaN
4,User-ID: 33333 loves the product!,33333


*(Notice how it completely ignored all the surrounding text and perfectly pulled out the IDs? For row 3, it returned `NaN` because it correctly realized there was no 5-digit number in the text!)*

# 4. Fuzzy Matching (Handling Typos)
Even after lowercasing our cities, we still have a problem: `"new york"` and `"new yrk"`. One is a typo. 

To fix typos programmatically, Data Scientists use **Fuzzy Matching** (often calculating the *Levenshtein Distance*). It measures how many single-character edits (insertions, deletions, substitutions) it takes to change one word into another.

*Note: In the real world, you would install a library called `thefuzz` for this. Since we are in a pure Pandas environment, let's write a simple mapping function to handle our known typos!*

In [5]:
# A manual mapping dictionary to fix known typos and abbreviations
city_corrections = {
    'new yrk': 'new york',
    'l.a.': 'los angeles'
}

# .replace() looks at the dictionary. If it finds the key, it swaps it for the value.
df_clean['city'] = df_clean['city'].replace(city_corrections)

print("--- Data after Typo Correction ---")
display(df_clean[['customer_name', 'city']])

--- Data after Typo Correction ---


,customer_name,city
0,Alice,new york
1,Bob,new york
2,Charlie,new york
3,David,los angeles
4,Eve,los angeles


## Real-World Use Case or Analogy:
Think of Text Cleaning like **Processing Mail at the Post Office**:

* **Basic Standardization (Lowercasing & Stripping)**: A machine scans the envelopes. Some people write in ALL CAPS, some in cursive. The machine forces all the text into a standard, computer-readable block font so it can actually be sorted.
* **Fuzzy Matching (Typos)**: Someone writes "Washngton D.C." on the envelope. The computer knows that "Washngton" is not a real place, but it calculates that it is only one letter away from "Washington". It automatically fixes the typo and routes the mail correctly instead of throwing it in the trash.
* **Regex (Pattern Extraction)**: The post office doesn't actually read your life story written on the back of the envelope. It uses a specific laser (Regex) designed to *only* look for a 5-digit pattern at the very bottom right corner (The Zip Code). It ignores all other text.
* **The "Other" Bucket (Rare Categories)**: The post office has dedicated trucks for major cities like New York, Chicago, and LA. But they get exactly one letter a year going to "Monowi, Nebraska" (Population: 1). They don't buy a dedicated truck for Monowi. They put that letter into a generic "Midwest-Other" bin to be sorted later, saving the Post Office millions of dollars in logistics.

---